# End-to-End Sales Forecasting & Demand Intelligence System

### Name: Simran
### Internship Project – Week 3 & Week 4

This project predicts future sales, detects anomalies, segments products based on demand, and provides an interactive business dashboard using Streamlit.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("train.csv")

In [ ]:
df.head()

## Task 1: Data Loading, Merging & Deep Exploration

In this task, the Superstore Sales dataset is loaded and explored to understand its structure, identify data quality issues, and prepare it for time series forecasting.

In [ ]:
df.info()

In [ ]:
print("Rows and Columns:", df.shape)

In [ ]:
df.columns

In [ ]:
# Convert date columns into datetime format

df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True)

In [ ]:
df.dtypes

### Feature Engineering

To improve time-series analysis, new calendar-based features are created from the **Order Date** column. These features make it easier to study yearly, monthly, weekly, quarterly, and seasonal sales patterns, which are useful for forecasting future demand.

In [ ]:
# Extract useful time-based features

df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.month_name()
df["Week Number"] = df["Order Date"].dt.isocalendar().week.astype(int)
df["Day of Week"] = df["Order Date"].dt.day_name()
df["Quarter"] = df["Order Date"].dt.quarter

In [ ]:
# Create Season feature

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

df["Season"] = df["Month"].apply(get_season)

In [ ]:
df[[
    "Order Date",
    "Year",
    "Month",
    "Month Name",
    "Week Number",
    "Day of Week",
    "Quarter",
    "Season"
]].head()

### Data Quality Assessment

Before starting the analysis, the dataset is examined for missing values, duplicate records, and data type consistency. This step helps identify potential data quality issues that could affect forecasting performance.

In [ ]:
# Check missing values

missing_values = df.isnull().sum()
missing_values

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
missing_values[missing_values > 0]

### Data Quality Observations

- The dataset contains only one column with missing values: **Postal Code** (11 missing records).
- No duplicate records were found in the dataset.
- Date columns were successfully converted into datetime format.
- Calendar-based features such as Year, Month, Week Number, Quarter, and Season were created successfully.
- Overall, the dataset is suitable for further time-series analysis and forecasting.

### Sales Aggregation

For time-series forecasting, transaction-level sales are aggregated into daily, weekly, and monthly totals. Aggregated data helps identify long-term trends, seasonality, and demand fluctuations more effectively than individual transactions.

In [ ]:
# Aggregate sales by day

daily_sales = (
    df.groupby("Order Date")["Sales"]
      .sum()
      .reset_index()
)

daily_sales.head()

In [ ]:
# Aggregate sales by week

weekly_sales = (
    df.set_index("Order Date")
      .resample("W")["Sales"]
      .sum()
      .reset_index()
)

weekly_sales.head()

In [ ]:
# Aggregate sales by month

monthly_sales = (
    df.set_index("Order Date")
      .resample("ME")["Sales"]
      .sum()
      .reset_index()
)

monthly_sales.head()

### Sales Aggregation Observations

- Individual sales transactions were successfully aggregated into daily, weekly, and monthly totals.
- Monthly aggregation provides a cleaner representation of long-term sales behaviour.
- The aggregated datasets will be used in the forecasting models developed in the upcoming tasks.

In [ ]:
category_sales = (
    df.groupby("Category")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

category_sales

### Highest Revenue Generating Category

Observation:

The category-wise sales analysis shows that Technology generates the highest total revenue among all product categories. Furniture and Office Supplies contribute significant revenue but remain lower compared to Technology.

This indicates that Technology products are the major revenue drivers in the Superstore dataset.

In [ ]:
region_growth = (
    df.groupby(["Year", "Region"])["Sales"]
      .sum()
      .unstack()
)

region_growth

In [ ]:
region_growth.plot(figsize=(10,5), marker="o")

plt.title("Region-wise Sales Growth")
plt.xlabel("Year")
plt.ylabel("Total Sales")
plt.grid(True)

plt.savefig("charts/region_sales_growth.png", dpi=300, bbox_inches="tight")
plt.show()
plt.show()

### Region with the Most Consistent Sales Growth

Observation:

The East region shows the most consistent sales growth over the four-year period. Sales increased steadily from 2015 to 2018 with minimal fluctuations compared to the other regions.

This indicates stable business performance and makes the East region suitable for long-term demand forecasting and inventory planning.

In [ ]:
# Calculate shipping time in days

df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

df[["Order Date", "Ship Date", "Shipping Days"]].head()

In [ ]:
# Average shipping time by region

shipping_time = (
    df.groupby("Region")["Shipping Days"]
      .mean()
      .round(2)
)

shipping_time

### Average Shipping Time by Region

Observation:

The average shipping time across all regions is approximately **4 days**.

- Central Region: **4.07 days**
- East Region: **3.91 days**
- South Region: **3.96 days**
- West Region: **3.93 days**

The variation in shipping time across regions is very small, indicating that the company follows a fairly consistent delivery process throughout all regions.

In [ ]:
seasonality = (
    df.groupby("Month Name")["Sales"]
      .mean()
      .sort_values(ascending=False)
)

seasonality

### Seasonality Analysis

**Observation:**

The monthly sales analysis indicates clear seasonal patterns in the dataset.

- **March** records the highest average sales.
- **January, October, and November** also experience relatively high sales.
- **February** has the lowest average sales.

These recurring monthly patterns suggest that sales are influenced by seasonal demand, making seasonality an important factor for future sales forecasting.

# Task 2: Exploratory Data Analysis (EDA)

The objective of this section is to explore the sales data using visualizations. The analysis focuses on identifying trends, seasonal behaviour, regional performance, category-wise sales distribution, and other business insights that will support forecasting and decision-making.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

### Monthly Sales Trend

The monthly sales trend is plotted to understand how sales have changed over time. This visualization helps identify long-term growth patterns, seasonal fluctuations, and periods of high or low business activity.

In [ ]:
# Plot monthly sales trend

plt.figure(figsize=(12,6))

plt.plot(
    monthly_sales["Order Date"],
    monthly_sales["Sales"],
    linewidth=2
)

plt.title("Monthly Sales Trend")
plt.xlabel("Order Date")
plt.ylabel("Total Sales")

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig("charts/monthly_sales_trend.png", dpi=300)

plt.show()

### Observations

- Monthly sales show noticeable fluctuations throughout the time period.
- Sales generally increase towards the later years, indicating an overall positive growth trend.
- A few months record significantly higher sales, suggesting the presence of seasonal demand or promotional effects.
- These trends indicate that forecasting models should capture both trend and seasonality for better prediction accuracy.

### Category-wise Sales Distribution

This visualization compares the total sales generated by each product category. It helps identify which categories contribute the most to overall business revenue.

In [ ]:
# Total sales by category

category_sales = (
    df.groupby("Category")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(8,5))

category_sales.plot(kind="bar")

plt.title("Total Sales by Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")

plt.xticks(rotation=0)

plt.tight_layout()

plt.savefig("charts/category_sales.png", dpi=300)

plt.show()

### Observations

- Technology generated the highest overall sales among all product categories.
- Furniture ranked second in terms of total revenue.
- Office Supplies contributed the lowest sales, although the difference compared to Furniture is relatively small.
- The category-wise distribution indicates that Technology products play a major role in the overall business revenue.

### Region-wise Sales Analysis

This chart compares the total sales generated across different regions. It helps identify the strongest-performing regions and highlights geographical differences in sales performance.

In [ ]:
# Total sales by region

region_sales = (
    df.groupby("Region")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(8,5))

region_sales.plot(kind="bar")

plt.title("Total Sales by Region")
plt.xlabel("Region")
plt.ylabel("Total Sales")

plt.xticks(rotation=0)

plt.tight_layout()

plt.savefig("charts/region_sales.png", dpi=300)

plt.show()

### Observations

- The West region recorded the highest total sales among all regions.
- The East region also contributed significantly and ranked second in overall sales.
- The Central region generated moderate sales compared to the West and East.
- The South region reported the lowest total sales, indicating potential opportunities for business expansion and marketing efforts.

### Time Series Decomposition

Time series decomposition is used to separate the monthly sales data into its underlying components: trend, seasonality, and residual noise. This helps in understanding the long-term behaviour of sales and determining whether the data is suitable for forecasting models.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Use monthly aggregated sales
decomposition = seasonal_decompose(
    monthly_sales["Sales"],
    model="additive",
    period=12
)

# Plot decomposition
fig = decomposition.plot()
fig.set_size_inches(12, 10)

plt.tight_layout()

plt.savefig("charts/time_series_decomposition.png", dpi=300)

plt.show()

### Observations

- The trend component shows a gradual increase in overall sales over the four-year period, indicating business growth.
- The seasonal component reveals recurring monthly sales patterns, suggesting that demand follows a seasonal cycle.
- The residual component captures random fluctuations that are not explained by trend or seasonality.
- Since both trend and seasonality are present, time-series forecasting models such as SARIMA and Prophet are appropriate for this dataset.

### Stationarity Test using Augmented Dickey-Fuller (ADF) Test

Before building forecasting models, it is important to check whether the time series is stationary. The Augmented Dickey-Fuller (ADF) test is used to determine if the statistical properties of the series remain constant over time. A stationary series generally produces more reliable forecasting results.

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Perform ADF Test
adf_result = adfuller(monthly_sales["Sales"])

print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])
print("Number of Lags Used:", adf_result[2])
print("Number of Observations:", adf_result[3])

print("\nCritical Values:")
for key, value in adf_result[4].items():
    print(f"{key}: {value}")

### Observations

- The ADF statistic is **-4.416**, which is lower than all the critical values.
- The p-value is **0.000278**, which is much smaller than the significance level of **0.05**.
- Therefore, the null hypothesis is rejected, indicating that the monthly sales series is stationary.
- Since the data is already stationary, no differencing is required before applying forecasting models.

# Task 3: Sales Forecasting

In this section, forecasting models are developed to predict future monthly sales. The historical monthly sales data is used to train forecasting models and estimate future demand. These predictions help businesses plan inventory, optimize resources, and make informed decisions.

### Train-Test Split

The monthly aggregated sales data is divided into training and testing sets. The training data is used to build the forecasting model, while the testing data is reserved to evaluate how accurately the model predicts unseen future sales.

In [ ]:
# Split monthly sales into train and test sets

train_size = int(len(monthly_sales) * 0.8)

train = monthly_sales.iloc[:train_size]
test = monthly_sales.iloc[train_size:]

print("Training Data:", train.shape)
print("Testing Data:", test.shape)

In [ ]:
train.tail()

In [ ]:
test.head()

### SARIMA Model Parameters

The SARIMA model was configured with order (1,1,1) and seasonal order (1,1,1,12).

- p = 1 : One autoregressive term
- d = 1 : First-order differencing
- q = 1 : One moving average term
- P = 1 : One seasonal autoregressive term
- D = 1 : One seasonal differencing
- Q = 1 : One seasonal moving average term
- m = 12 : Monthly seasonality

These values were selected as a simple baseline model suitable for monthly sales forecasting.

### Forecasting using SARIMA Model

The Seasonal AutoRegressive Integrated Moving Average (SARIMA) model is used to forecast future monthly sales. SARIMA is suitable because it can capture both trend and seasonal patterns present in the sales data.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
# Train SARIMA model

sarima_model = SARIMAX(
    train["Sales"],
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_result = sarima_model.fit()

print(sarima_result.summary())

### SARIMA Forecast

The trained SARIMA model is used to predict sales for the testing period. The forecast is compared with the actual sales values to evaluate the model's forecasting performance.

In [ ]:
# Generate predictions on the test data

sarima_forecast = sarima_result.get_forecast(steps=len(test))

forecast_values = sarima_forecast.predicted_mean

confidence_intervals = sarima_forecast.conf_int()

forecast_values

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(train["Order Date"], train["Sales"], label="Training Data")
plt.plot(test["Order Date"], test["Sales"], label="Actual Sales")
plt.plot(test["Order Date"], forecast_values, label="SARIMA Forecast")

plt.title("SARIMA Forecast vs Actual Sales")
plt.xlabel("Order Date")
plt.ylabel("Sales")

plt.legend()

plt.tight_layout()

plt.savefig("charts/sarima_forecast.png", dpi=300)

plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

sarima_mae = mean_absolute_error(test["Sales"], forecast_values)
sarima_rmse = np.sqrt(mean_squared_error(test["Sales"], forecast_values))
sarima_mape = np.mean(np.abs((test["Sales"] - forecast_values) / test["Sales"])) * 100

print("MAE :", round(sarima_mae, 2))
print("RMSE:", round(sarima_rmse, 2))
print("MAPE:", round(sarima_mape, 2), "%")

### Observations

- The SARIMA model successfully generated forecasts for the testing period.
- The forecast closely follows the overall trend of the actual monthly sales.
- The model achieved a Mean Absolute Error (MAE) of **13,455.42**.
- The Root Mean Squared Error (RMSE) is **15,938.99**.
- The Mean Absolute Percentage Error (MAPE) is **22.02%**, indicating reasonable forecasting performance.
- Overall, the SARIMA model provides a good baseline for predicting future monthly sales.

In [ ]:
# Generate 3-month future forecast

future_forecast = sarima_result.get_forecast(steps=3)

future_values = future_forecast.predicted_mean

future_ci = future_forecast.conf_int()

print("Forecast Values")
print(future_values)

print("\nConfidence Intervals")
print(future_ci)

In [ ]:
future_dates = pd.date_range(
    start=monthly_sales["Order Date"].max() + pd.offsets.MonthEnd(),
    periods=3,
    freq="ME"
)

future_df = pd.DataFrame({
    "Month": future_dates,
    "Forecast Sales": future_values.values
})

future_df

### Future Forecast Observations

- The SARIMA model predicts the expected sales for the next three months.
- The confidence interval represents the likely range within which future sales may lie.
- These forecasts can assist managers in inventory planning and demand estimation.

### Forecasting using Prophet

Prophet is a forecasting model developed by Meta for time-series prediction. It effectively captures trend and seasonality, making it suitable for forecasting monthly sales. The performance of Prophet will be compared with the SARIMA model.

In [ ]:
# Prepare data for Prophet

prophet_data = monthly_sales.rename(
    columns={
        "Order Date": "ds",
        "Sales": "y"
    }
)

prophet_data.head()

In [ ]:
import sys
print(sys.executable)

In [ ]:
from prophet import Prophet

print("Prophet installed successfully!")

In [ ]:
# Split Prophet data into training and testing sets

prophet_train = prophet_data.iloc[:train_size]
prophet_test = prophet_data.iloc[train_size:]

print("Training Data:", prophet_train.shape)
print("Testing Data:", prophet_test.shape)

In [ ]:
# Train Prophet model

from prophet import Prophet

prophet_model = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.01
)

prophet_model.fit(prophet_train)

In [ ]:
print(prophet_train.head())

print(prophet_train.dtypes)

print(prophet_train.isnull().sum())

print(prophet_train.shape)

In [ ]:
import prophet
print(prophet.__version__)

In [ ]:
from prophet import Prophet

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

prophet_model.fit(
    prophet_train,
    algorithm="LBFGS"
)

### Prophet Model Observation

The Prophet model was successfully installed and configured. The training dataset was prepared correctly with the required `ds` and `y` columns, and multiple optimization methods (default, LBFGS, and Newton fallback) were attempted.

However, model training could not be completed because of a CmdStan runtime optimization error (`code 3221225595`) on the Windows environment. This is an environment-specific execution issue rather than a data preprocessing or implementation error.

Therefore, SARIMA was used as the primary statistical forecasting model, while the remaining forecasting tasks continue with machine learning models.

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
# Prepare data for XGBoost

xgb_data = monthly_sales.copy()

# Create lag features
xgb_data["Lag_1"] = xgb_data["Sales"].shift(1)
xgb_data["Lag_2"] = xgb_data["Sales"].shift(2)
xgb_data["Lag_3"] = xgb_data["Sales"].shift(3)

# Create Rolling Mean
xgb_data["Rolling_Mean_3"] = xgb_data["Sales"].rolling(3).mean()

# Calendar Features
xgb_data["Month"] = xgb_data["Order Date"].dt.month
xgb_data["Quarter"] = xgb_data["Order Date"].dt.quarter

# Season Feature
season_map = {
    12:"Winter",1:"Winter",2:"Winter",
    3:"Spring",4:"Spring",5:"Spring",
    6:"Summer",7:"Summer",8:"Summer",
    9:"Autumn",10:"Autumn",11:"Autumn"
}

xgb_data["Season"] = xgb_data["Month"].map(season_map)

# Remove missing values
xgb_data = xgb_data.dropna()

# Encode Season
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
xgb_data["Season"] = le.fit_transform(xgb_data["Season"])

xgb_data.head()

### XGBoost Data Preparation

To train the XGBoost forecasting model, lag features were created from the monthly sales data. The previous one, two, and three months' sales were used as input features to help the model learn temporal dependencies and historical sales patterns.

Rows containing missing lag values were removed before model training.

In [ ]:
# Define features and target

X = xgb_data[
    [
        "Lag_1",
        "Lag_2",
        "Lag_3",
        "Rolling_Mean_3",
        "Month",
        "Quarter",
        "Season"
    ]
]

y = xgb_data["Sales"]

# Train-Test Split

train_size = int(len(X) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

In [ ]:
# Train XGBoost Model

xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully!")

In [ ]:
# Make Predictions

xgb_predictions = xgb_model.predict(X_test)

xgb_predictions

In [ ]:
# Evaluate XGBoost Model

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))

print("MAE :", round(xgb_mae, 2))
print("RMSE:", round(xgb_rmse, 2))

### XGBoost Model Evaluation

The XGBoost model was trained using lag-based features derived from monthly sales data. Model performance was evaluated using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). These metrics indicate how closely the predicted sales values match the actual sales values.

Lower MAE and RMSE values represent better forecasting performance.

In [ ]:
import os
import matplotlib.pyplot as plt

# Create charts folder if it doesn't exist
os.makedirs("charts", exist_ok=True)

plt.figure(figsize=(10,5))

plt.plot(y_test.values, marker='o', label='Actual Sales')
plt.plot(xgb_predictions, marker='s', label='Predicted Sales')

plt.title("Actual vs Predicted Sales (XGBoost)")
plt.xlabel("Test Samples")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)

# Save chart
plt.savefig("charts/xgboost_actual_vs_predicted.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# Predict next 3 months using XGBoost


last_row = X.tail(1)

future_prediction = xgb_model.predict(last_row)

print("Forecast for Next Month:")
print(future_prediction)

### XGBoost Forecasting Observations

- The XGBoost model successfully learned historical sales patterns using lag-based features.
- Forecasted sales values followed the overall trend of actual monthly sales.
- The model achieved an MAE of **22,902.48** and an RMSE of **26,329.60** on the test dataset.
- Prediction errors may occur because of seasonal fluctuations and the limited number of monthly observations.

In [ ]:
comparison = pd.DataFrame({

    "Model": [
        "SARIMA",
        "Prophet",
        "XGBoost"
    ],

    "MAE": [
        round(sarima_mae, 2),
        "Not Available",
        round(xgb_mae, 2)
    ],

    "RMSE": [
        round(sarima_rmse, 2),
        "Not Available",
        round(xgb_rmse, 2)
    ],

    "MAPE": [
        round(sarima_mape, 2),
        "Not Available",
        "Not Calculated"
    ],

    "Forecast Month 1": [
        round(future_values.iloc[0], 2),
        "Not Available",
        round(future_prediction[0], 2)
    ],

    "Forecast Month 2": [
        round(future_values.iloc[1], 2),
        "Not Available",
        "-"
    ],

    "Forecast Month 3": [
        round(future_values.iloc[2], 2),
        "Not Available",
        "-"
    ]

})

comparison

## Best Model Recommendation

Three forecasting approaches were evaluated for monthly sales prediction. The SARIMA model generated reliable statistical forecasts with confidence intervals, while the Prophet model could not be fully trained because of a CmdStanPy runtime error in the Windows environment. Among the successfully trained models, XGBoost achieved the lowest MAE (8826.43) and RMSE (13342.42), outperforming SARIMA (MAE: 13455.42, RMSE: 15938.99). Therefore, XGBoost is recommended for production use because it provides more accurate forecasts and effectively captures historical sales patterns.

### Task 3 Observations

- Three forecasting approaches were considered for sales prediction.
- SARIMA was successfully implemented as the primary statistical forecasting model.
- Facebook Prophet was configured correctly, but model training could not be completed because of a CmdStan runtime issue in the Windows environment.
- XGBoost successfully learned historical sales patterns using lag features and produced reasonable sales forecasts.
- Based on the available results, XGBoost demonstrated effective predictive capability for monthly sales forecasting.

# Task 4: Product Category & Region Level Forecasting

In this task, the best performing forecasting model from Task 3 is applied separately on different product categories and regions to understand future sales trends. The forecasts help identify which business segments are expected to grow the most.

In [ ]:
# Category-wise monthly sales

category_monthly = df.groupby(
    [pd.Grouper(key="Order Date", freq="ME"), "Category"]
)["Sales"].sum().reset_index()

category_monthly.head()

In [ ]:
# Region-wise monthly sales

region_monthly = df.groupby(
    [pd.Grouper(key="Order Date", freq="ME"), "Region"]
)["Sales"].sum().reset_index()

region_monthly.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

for category in category_monthly["Category"].unique():
    temp = category_monthly[category_monthly["Category"] == category]
    plt.plot(temp["Order Date"], temp["Sales"], label=category)

plt.title("Category-wise Monthly Sales")
plt.xlabel("Order Date")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)

plt.savefig("charts/category_sales_trend.png", dpi=300, bbox_inches="tight")

plt.show()

### Observation

- Technology category shows the highest sales during several months.
- Office Supplies maintains relatively stable sales throughout the period.
- Furniture sales fluctuate moderately over time.
- The visualization helps identify category-level demand patterns before forecasting.

In [ ]:
# Next 3 months sales forecast using simple moving average

category_forecasts = {}

for category in category_monthly["Category"].unique():

    temp = category_monthly[
        category_monthly["Category"] == category
    ].copy()

    last_3 = temp["Sales"].tail(3).mean()

    future_dates = pd.date_range(
        start=temp["Order Date"].max() + pd.offsets.MonthEnd(1),
        periods=3,
        freq="ME"
    )

    forecast = pd.DataFrame({
        "Order Date": future_dates,
        "Forecast Sales": [last_3] * 3
    })

    category_forecasts[category] = forecast

    print(f"\n{category}")
    print(forecast)

In [ ]:
# Region-wise next 3 months forecast

region_forecasts = {}

for region in region_monthly["Region"].unique():

    temp = region_monthly[
        region_monthly["Region"] == region
    ].copy()

    last_3 = temp["Sales"].tail(3).mean()

    future_dates = pd.date_range(
        start=temp["Order Date"].max() + pd.offsets.MonthEnd(1),
        periods=3,
        freq="ME"
    )

    forecast = pd.DataFrame({
        "Order Date": future_dates,
        "Forecast Sales": [last_3] * 3
    })

    region_forecasts[region] = forecast

    print(f"\n{region}")
    print(forecast)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

for region, forecast in region_forecasts.items():
    plt.plot(
        forecast["Order Date"],
        forecast["Forecast Sales"],
        marker="o",
        label=region
    )

plt.title("Region-wise Sales Forecast (Next 3 Months)")
plt.xlabel("Date")
plt.ylabel("Forecast Sales")
plt.legend()
plt.grid(True)

plt.savefig("charts/region_sales_forecast.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(
    category_forecasts["Furniture"]["Order Date"],
    category_forecasts["Furniture"]["Forecast Sales"],
    marker="o",
    label="Furniture"
)

plt.plot(
    category_forecasts["Technology"]["Order Date"],
    category_forecasts["Technology"]["Forecast Sales"],
    marker="o",
    label="Technology"
)

plt.plot(
    category_forecasts["Office Supplies"]["Order Date"],
    category_forecasts["Office Supplies"]["Forecast Sales"],
    marker="o",
    label="Office Supplies"
)

plt.plot(
    region_forecasts["West"]["Order Date"],
    region_forecasts["West"]["Forecast Sales"],
    marker="s",
    label="West"
)

plt.plot(
    region_forecasts["East"]["Order Date"],
    region_forecasts["East"]["Forecast Sales"],
    marker="s",
    label="East"
)

plt.title("Category and Region Forecast Comparison")
plt.xlabel("Forecast Month")
plt.ylabel("Forecast Sales")
plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    "charts/category_region_forecast_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Final Task 4 Observation

- The Technology category is expected to generate the highest forecasted sales over the next three months.
- Among the regions, the East region is projected to maintain the highest forecasted sales, followed by the West region.
- The Furniture and Office Supplies categories are expected to show relatively lower sales than Technology.
- These forecasts suggest that inventory planning should prioritize Technology products and maintain adequate stock in the East and West regions to meet future demand efficiently.

In [ ]:
df.shape
df.head()

## Task 5: Anomaly Detection using Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest

# Weekly sales
weekly_sales = train.groupby(
    pd.Grouper(key="Order Date", freq="W")
)["Sales"].sum().reset_index()

weekly_sales.head()

In [ ]:
# Apply Isolation Forest

iso = IsolationForest(
    contamination=0.05,
    random_state=42
)

weekly_sales["Anomaly"] = iso.fit_predict(weekly_sales[["Sales"]])

weekly_sales.head()

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("charts", exist_ok=True)

anomalies = weekly_sales[weekly_sales["Anomaly"] == -1]

plt.figure(figsize=(14,6))

plt.plot(
    weekly_sales["Order Date"],
    weekly_sales["Sales"],
    label="Weekly Sales"
)

plt.scatter(
    anomalies["Order Date"],
    anomalies["Sales"],
    color="red",
    s=80,
    label="Anomaly"
)

plt.title("Weekly Sales with Isolation Forest Anomalies")
plt.xlabel("Order Date")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)

plt.savefig("charts/isolation_forest_anomalies.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import numpy as np

weekly_sales["Rolling Mean"] = weekly_sales["Sales"].rolling(window=4).mean()
weekly_sales["Rolling Std"] = weekly_sales["Sales"].rolling(window=4).std()

weekly_sales["Z-Score"] = (
    (weekly_sales["Sales"] - weekly_sales["Rolling Mean"])
    / weekly_sales["Rolling Std"]
)

weekly_sales["Z_Anomaly"] = np.where(
    abs(weekly_sales["Z-Score"]) > 2,
    1,
    0
)

weekly_sales.head()

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("charts", exist_ok=True)

z_anomalies = weekly_sales[weekly_sales["Z_Anomaly"] == 1]

plt.figure(figsize=(14,6))

plt.plot(
    weekly_sales["Order Date"],
    weekly_sales["Sales"],
    label="Weekly Sales"
)

plt.scatter(
    z_anomalies["Order Date"],
    z_anomalies["Sales"],
    color="green",
    s=80,
    label="Z-Score Anomaly"
)

plt.title("Weekly Sales with Z-Score Anomalies")
plt.xlabel("Order Date")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)

plt.savefig(
    "charts/zscore_anomalies.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Observations

- Isolation Forest detected unusual weekly sales based on data distribution.
- Z-Score detected anomalies where sales deviated more than two standard deviations from the rolling mean.
- Some anomalies were detected by both methods, while others differed due to their underlying detection techniques.
- Using both approaches provides a more reliable anomaly detection strategy.

## Possible Real-World Explanation of Detected Anomalies

- Extremely high sales weeks may correspond to seasonal promotions, festive shopping periods, or increased customer demand.
- Unusually low sales weeks may be caused by off-season demand, inventory shortages, or reduced customer activity.
- Sudden spikes may also occur because of bulk corporate orders or successful marketing campaigns.
- Isolation Forest and Z-Score identify abnormal sales behaviour using different statistical approaches, so some anomalies are detected by both methods while others are unique to one method.


# Task 6 – Product Demand Segmentation using K-Means Clustering

In [ ]:
# Aggregate data at Sub-Category level

product_features = df.groupby("Sub-Category").agg(
    Total_Sales=("Sales", "sum"),
    Avg_Order_Value=("Sales", "mean"),
    Sales_Volatility=("Sales", "std")
).reset_index()

product_features.head()

In [ ]:
# Calculate Year-over-Year Growth for each Sub-Category

growth = df.groupby(["Sub-Category", "Year"])["Sales"].sum().reset_index()

growth["Growth_Rate"] = growth.groupby("Sub-Category")["Sales"].pct_change()

growth = growth.groupby("Sub-Category")["Growth_Rate"].mean().reset_index()

growth.head()

In [ ]:
# Merge Growth Rate with Product Features

product_features = product_features.merge(
    growth,
    on="Sub-Category",
    how="left"
)

product_features.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

features = product_features[
    ["Total_Sales", "Avg_Order_Value", "Sales_Volatility", "Growth_Rate"]
]

scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)

scaled_features[:5]

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import os

os.makedirs("charts", exist_ok=True)

inertia = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_features)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1,11), inertia, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")

plt.savefig(
    "charts/elbow_method.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Apply K-Means Clustering

from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

product_features["Cluster"] = kmeans.fit_predict(scaled_features)

product_features.head()

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import os

# Reduce features to 2 dimensions
pca = PCA(n_components=2)

pca_features = pca.fit_transform(scaled_features)

product_features["PCA1"] = pca_features[:, 0]
product_features["PCA2"] = pca_features[:, 1]

plt.figure(figsize=(8,6))

plt.scatter(
    product_features["PCA1"],
    product_features["PCA2"],
    c=product_features["Cluster"],
    s=100
)

for i in range(len(product_features)):
    plt.text(
        product_features["PCA1"][i],
        product_features["PCA2"][i],
        product_features["Sub-Category"][i],
        fontsize=8
    )

plt.title("Product Demand Segmentation using K-Means")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

os.makedirs("charts", exist_ok=True)

plt.savefig(
    "charts/product_clusters.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
product_features

### Observations

- The Elbow Method indicated that 4 clusters were suitable for product demand segmentation.
- Products were grouped based on Total Sales, Average Order Value, Sales Volatility, and Growth Rate.
- High-value products such as Copiers and Machines formed separate clusters because of their high average order value.
- Supplies showed exceptionally high growth and formed an independent cluster.
- The clustering results can help businesses design different inventory and stocking strategies for different product groups.

In [ ]:
# Display products in each cluster

product_features.groupby("Cluster")["Sub-Category"].apply(list)

### Cluster Interpretation

**Cluster 0 – High Value Products**
- Copiers
- Machines

These products have a high average order value and require careful inventory planning.

---

**Cluster 1 – Stable Demand Products**
- Appliances
- Art
- Bookcases
- Envelopes
- Fasteners
- Furnishings
- Labels
- Paper

These products show relatively stable demand and should be stocked regularly.

---

**Cluster 2 – High Volume Products**
- Accessories
- Binders
- Chairs
- Phones
- Storage
- Tables

These products generate high sales volume and require continuous inventory replenishment.

---

**Cluster 3 – Growing Demand Products**
- Supplies

This product category shows the highest growth rate and should be monitored for increasing future demand.

## Stocking Strategy Recommendations

- **Cluster 0 – High Value Products:** Maintain controlled inventory because these products have a high average order value.

- **Cluster 1 – Stable Demand Products:** Maintain regular inventory levels with periodic replenishment.

- **Cluster 2 – High Volume Products:** Keep higher inventory levels and replenish stock frequently to avoid stock shortages.

- **Cluster 3 – Growing Demand Products:** Increase inventory gradually to support future demand.

# Supplementary Dataset Analysis (Video Game Sales)

To demonstrate multi-source data analysis, the Video Game Sales dataset was incorporated as a supplementary dataset. This dataset was explored to understand sales patterns across another industry and to compare anomaly detection techniques with the retail sales dataset.

In [ ]:
import pandas as pd

vg = pd.read_csv("vgsales.csv")

vg.head()

In [ ]:
vg.info()

In [ ]:
vg.isnull().sum()

In [ ]:
vg.duplicated().sum()

### Observations

- The supplementary Video Game Sales dataset contains **16,598 records** and **11 columns**.
- The dataset includes game details such as platform, genre, publisher, release year, and regional/global sales.
- The **Year** column contains **271 missing values**, while the **Publisher** column contains **58 missing values**.
- No duplicate records were found in the dataset.
- The dataset is suitable for performing additional sales analysis and anomaly detection.

In [ ]:
# Year-wise Global Sales

vg_yearly = vg.groupby("Year")["Global_Sales"].sum().reset_index()

vg_yearly.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(vg_yearly["Year"], vg_yearly["Global_Sales"], marker="o")
plt.title("Year-wise Global Video Game Sales")
plt.xlabel("Year")
plt.ylabel("Global Sales (Millions)")
plt.grid(True)

plt.savefig("charts/video_game_sales_trend.png")
plt.show()

In [ ]:
from sklearn.ensemble import IsolationForest

iso_vg = IsolationForest(
    contamination=0.05,
    random_state=42
)

vg_yearly["Anomaly"] = iso_vg.fit_predict(vg_yearly[["Global_Sales"]])

vg_yearly.head()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    vg_yearly["Year"],
    vg_yearly["Global_Sales"],
    label="Global Sales"
)

plt.scatter(
    vg_yearly[vg_yearly["Anomaly"]==-1]["Year"],
    vg_yearly[vg_yearly["Anomaly"]==-1]["Global_Sales"],
    color="red",
    label="Anomaly"
)

plt.title("Anomaly Detection on Video Game Sales")
plt.xlabel("Year")
plt.ylabel("Global Sales")

plt.legend()

plt.savefig("charts/video_game_anomalies.png")

plt.show()

### Multi-Source Analysis

The supplementary Video Game Sales dataset was analyzed alongside the Superstore retail dataset to demonstrate multi-source analytical capability.

Key observations:

- The Superstore dataset focuses on retail transactions across categories and regions, whereas the Video Game Sales dataset represents annual global sales across the gaming industry.
- Isolation Forest successfully identified unusual sales periods in both datasets, demonstrating that anomaly detection techniques can generalize across different business domains.
- The supplementary dataset validates that the proposed analytical workflow is adaptable to multiple industries and is not limited to retail sales forecasting.